# Two ways of syncing: assign to data, or explicit $emit

Every trait of a `VueTemplate` is two-way bound Vue data: the template can
assign to it directly and the change syncs back to Python.

In [ ]:
import traitlets
from ipyvue import VueTemplate


class DataCounter(VueTemplate):
    template = traitlets.Unicode(
        """
        <template>
            <button @click="count = count + 1">
                clicked {{ count }} times
            </button>
        </template>
    """
    ).tag(sync=True)
    count = traitlets.Int(0).tag(sync=True)


data_counter = DataCounter()
data_counter

In [ ]:
# the assignment in the template synced back to Python
data_counter.count

The template can also sync a trait **explicitly** with
`$emit("update:<name>", value)` (the same contract as vue's `.sync`
modifier / `v-model`), and send any event listed in `events` with
`$emit("<name>", value)`.

This style makes the data flow explicit (read the value, emit the change),
matches how pure Vue components are written, and ports cleanly to vue3's
`defineModel()` semantics.

In [ ]:
class EmitCounter(VueTemplate):
    template = traitlets.Unicode(
        """
        <template>
            <button @click="$emit('update:count', count + 1)">
                clicked {{ count }} times
            </button>
        </template>
    """
    ).tag(sync=True)
    count = traitlets.Int(0).tag(sync=True)


emit_counter = EmitCounter()
emit_counter

In [ ]:
# the $emit("update:count", ...) synced back to Python,
# and setting the trait still flows down into the prop
emit_counter.count = 10

Fully vue-like: with `template_props_support`, props declared in the
template's `<script>` block are honored — the trait arrives as a one-way
prop, assignment no longer syncs back (vue warns instead), only
`$emit("update:<name>", value)` does. Off by default because existing
templates rely on declared props being ignored.

In [ ]:
class PropsCounter(VueTemplate):
    template = traitlets.Unicode('''
        <template>
            <button @click="$emit('update:count', count + 1)">
                clicked {{ count }} times
            </button>
        </template>
        <script>
        export default {
            props: ["count"],
        };
        </script>
    ''').tag(sync=True)
    count = traitlets.Int(0).tag(sync=True)


props_counter = PropsCounter(template_props_support=True)
props_counter